In [4]:
%load_ext autoreload
%autoreload 2

import os
import sys
import optuna
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from omegaconf import OmegaConf

from sklearn.model_selection import KFold

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.ensembles import get_oof_catboost, get_oof_dnn, get_oof_lgbm

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
config = OmegaConf.load('../config/config.yaml')
    
train = pd.read_csv('../' + config.paths.train_path)
test = pd.read_csv('../' + config.paths.test_path)
        
target = config.target.column

X = train.drop(columns=target)
y = train[target]

if config.target.log_transform:
    y = np.log1p(y)

seed = 42
n_splits = 5

In [13]:

print('lgbm running')
lgbm_oof, lgbm_test = get_oof_lgbm(
    X=X,
    y=y,
    test=test,
    model_params=OmegaConf.to_container(config.models.lightgbm.params, resolve=True),
    n_splits=n_splits,
    seed=seed,
    use_total_sf=True,
    use_total_bathrooms=True,
    use_house_age=True
)

print('dnn running')
dnn_oof, dnn_test = get_oof_dnn(
    X=X,
    y=y,
    test=test,
    hidden_dim=config.models.dnn.params.hidden_dim,
    batch_size=config.models.dnn.training.batch_size,
    lr=config.models.dnn.training.lr,
    weight_decay=config.models.dnn.training.weight_decay,
    epochs=config.models.dnn.training.epochs,
    n_splits=n_splits,
    seed=seed,
    use_total_sf=True,
    use_total_bathrooms=True,
    use_house_age=True,
)

print('catboost running')
cat_oof, cat_test = get_oof_catboost(
    X=X,
    y=y,
    test=test,
    model_params=OmegaConf.to_container(config.models.catboost.params, resolve=True),
    n_splits=n_splits,
    seed=seed,
    use_total_sf=True,
    use_total_bathrooms=True,
    use_house_age=True
)

lgbm running
dnn running
catboost running


In [14]:
from sklearn.metrics import root_mean_squared_error

print(
    "lgbm OOF RMSE:",
    root_mean_squared_error(y, lgbm_oof)
)

print(
    "lgbm test shape:",
    lgbm_test.shape
)

print(
    "dnn OOF RMSE:",
    root_mean_squared_error(y, dnn_oof)
)

print(
    "dnn test shape:",
    dnn_test.shape
)

print(
    "CatBoost OOF RMSE:",
    root_mean_squared_error(y, cat_oof)
)

print(
    "CatBoost test shape:",
    cat_test.shape
)

lgbm OOF RMSE: 29873.676358110766
lgbm test shape: (1459,)
dnn OOF RMSE: 133937.17444280628
dnn test shape: (1459,)
CatBoost OOF RMSE: 28572.458058005646
CatBoost test shape: (1459,)
